In [5]:
# %% [markdown]
# ## ECB monetary-policy releases: merge all year folders -> clean
#
# The scraped releases live one-per-`.txt` inside per-year subfolders
# (`txt/1999/`, `txt/2000/`, ... `txt/2026/`), so this runs in two explicit steps:
#
#   STEP 1  MERGE  -> walk every year folder, read all raw .txt into ONE frame
#   STEP 2  CLEAN  -> isolate the monetary-policy body + parse the date
#
# After step 2 the frame has the SAME shape as the FOMC `docs` (`raw_text` +
# `doc_type`), so it drops straight into the downstream you already built
# (clean_text -> sentence split -> Gorodnichenko Panel A1/B1 filter -> holdout).
#
# Every scraped page has the same constant layout, 1999 through 2026:
#
#     [skip-links / search chrome / top nav menu]      <- junk (above the title)
#     PRESS RELEASE
#     Monetary policy decisions                        <- title (anchor)
#     <dd Month yyyy>                                  <- dateline (body starts AFTER)
#     ... the actual monetary-policy release ...       <- KEEP THIS
#     The President of the ECB will comment ...         <- procedural sign-off (body ENDS here)
#     Related topics / SEE ALSO / CONTACT / cookies    <- junk (below the body)

# %%
import re
from datetime import datetime
from pathlib import Path
import pandas as pd

# %% [markdown]
# ### !! Set this to wherever the "ECB Monetary policy decisions" folder lives !!
# `txt/` (the year subfolders) and `out/` are derived from it.

# %%
ECB_ROOT = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/"
                "ECB BRANCH/ECB Monetary policy decisions")

TXT_ROOT = ECB_ROOT / "txt"     # contains the 1999..2026 subfolders
OUT_DIR  = ECB_ROOT / "out"     # cleaned csv / jsonl written here

# Kept "statement" so this frame is compatible with your FOMC holdout sampler
# (it filters on doc_type == "statement"). Switch to "ecb_statement" only if you
# intend to CONCAT ECB with FOMC into one frame and want the banks separable.
ECB_DOC_TYPE = "statement"

# %% [markdown]
# ### Extraction patterns + functions

# %%
_MONTHS = (r"(?:January|February|March|April|May|June|July|August|"
           r"September|October|November|December)")

# The dateline directly under "Monetary policy decisions" — the handoff from page
# chrome to release body. "in the first half of 2000" won't match (needs
# day+month+year), so the first hit is always the real dateline.
ECB_DATE_RE = re.compile(rf"\b(\d{{1,2}}\s+{_MONTHS}\s+\d{{4}})\b", re.IGNORECASE)

# Body TERMINATORS — earliest one wins. The "President will comment ..." sign-off
# is a terminator, not noise: it's always the last line of the release, and on
# some pages (e.g. 6 June 2002) it's broken mid-sentence across three lines
# ("... at a" / "press conference" / "starting at 2.30 p.m. today"), so cutting
# here also removes those dangling fragments. Everything below (Related topics
# tag-list, SEE ALSO, CONTACT block, cookie/privacy notices) is page furniture.
# Anchored to line starts so a stray "contact" inside prose can't clip the body.
ECB_END_MARKERS = re.compile(
    r"^[ \t]*(?:"
    r"the president of the ecb will comment"     # procedural sign-off
    r"|related topics"                           # tag list (newer releases)
    r"|see also"                                 # 'find out more' block
    r"|contact[ \t]*$"                           # ECB address block header
    r"|media contacts"
    r"|disclaimer[ \t]*$"
    r"|are you happy with this page"
    r"|our website uses cookies"
    r")",
    re.IGNORECASE | re.MULTILINE,
)

# Scraped skip-links / search widget / top navigation menu items. Matched as
# WHOLE lines (exact, case-insensitive) so they can never clip real prose.
ECB_NAV_LINES = {
    "skip to:", "skip to", "navigation", "content", "footer",
    "anytime", "past month", "past year", "search options", "image preview",
    "home", "media", "explainers", "research & publications", "statistics",
    "monetary policy", "the \u20acuro", "payments & markets", "careers",
    "suggestions", "sort by", "relevance", "date", "press release",
    "monetary policy decisions",
}

# In-body leftover to drop line-wise (NOT a terminator): the '***' rule the ECB
# puts between the release and its closing paragraph is genuine mid-body markup,
# and the paragraph after it ("The Governing Council stands ready ...") is real
# content, so we delete only the separator line and keep going.
ECB_BODY_NOISE = re.compile(r"^\s*\*{2,}\s*$")


def extract_ecb_body(raw_text: str) -> str:
    """Return only the monetary-policy release text, stripped of page chrome."""
    if not isinstance(raw_text, str) or not raw_text.strip():
        return ""
    text = raw_text

    # 1) cut the footer at the earliest terminator
    end = ECB_END_MARKERS.search(text)
    if end:
        text = text[:end.start()]

    # 2) start after the dateline that follows the release title. Anchoring on the
    #    FIRST "monetary policy decisions" means a body that repeats the phrase
    #    (e.g. "took the following monetary policy decisions:") is left untouched.
    title = re.search(r"monetary policy decisions", text, re.IGNORECASE)
    if title:
        text = text[title.end():]
        d = ECB_DATE_RE.search(text)
        if d:
            text = text[d.end():]

    # 3) tidy surviving lines: drop nav tokens and '***' separators, keep prose
    kept = []
    for ln in text.splitlines():
        s = ln.strip()
        if not s or s.lower() in ECB_NAV_LINES or ECB_BODY_NOISE.match(s):
            continue
        kept.append(s)
    return "\n".join(kept).strip()


def parse_ecb_date(raw_text: str):
    """Dateline under the title, as YYYY-MM-DD (matches the FOMC date format)."""
    if not isinstance(raw_text, str):
        return None
    title = re.search(r"monetary policy decisions", raw_text, re.IGNORECASE)
    d = ECB_DATE_RE.search(raw_text, title.end() if title else 0)
    if not d:
        return None
    try:
        return datetime.strptime(d.group(1), "%d %B %Y").strftime("%Y-%m-%d")
    except ValueError:
        return None

# %% [markdown]
# ### STEP 1 - MERGE every year folder into one raw frame

# %%
_YEAR_RE = re.compile(r"(?:19|20)\d{2}")

def gather_ecb_txt(txt_root, pattern="*.txt") -> pd.DataFrame:
    """Recurse into every year subfolder and read each raw .txt into one frame.
    No cleaning yet, so you can eyeball the merged pile first. `source_file` is
    the path relative to txt_root (e.g. '2007/xxx.txt') so identically-named
    files in different year folders never collide."""
    txt_root = Path(txt_root)
    if not txt_root.exists():
        raise FileNotFoundError(f"{txt_root} not found — check ECB_ROOT.")
    rows = []
    for fp in sorted(txt_root.rglob(pattern)):        # rglob = descend into 1999..2026
        rel = fp.relative_to(txt_root).as_posix()
        ym = _YEAR_RE.search(rel)                     # year from the folder path
        rows.append({
            "source_file": rel,
            "folder_year": int(ym.group()) if ym else None,
            "raw_text_full": fp.read_text(encoding="utf-8", errors="replace"),
        })
    df = pd.DataFrame(rows, columns=["source_file", "folder_year", "raw_text_full"])
    print(f"[merge] {len(df)} raw .txt files from {txt_root}")
    return df

ecb_raw = gather_ecb_txt(TXT_ROOT)
ecb_raw.head()

# %% [markdown]
# ### STEP 2 - CLEAN the merged frame (isolate body + parse date)

# %%
def clean_ecb_frame(raw_df, doc_type=ECB_DOC_TYPE) -> pd.DataFrame:
    """Apply body isolation + date parse to the merged frame. Output is shaped
    like the FOMC `docs` (raw_text + doc_type) plus date / folder_year /
    source_file. Cross-checks dateline year vs folder year and warns on any
    mismatch (a misfiled or misparsed document)."""
    df = raw_df.copy()
    df["raw_text"]  = df["raw_text_full"].map(extract_ecb_body)
    df["date"]      = df["raw_text_full"].map(parse_ecb_date)
    df["doc_type"]  = doc_type
    df["date_year"] = df["date"].str.slice(0, 4).astype("Int64")

    # drop pages we couldn't isolate a body from (flag them, never guess)
    empty = df["raw_text"].str.len() == 0
    for f in df.loc[empty, "source_file"]:
        print(f"  [skip] no body extracted: {f}")
    df = df[~empty].copy()

    # folder-year vs dateline-year cross-check
    mism = (df["folder_year"].notna() & df["date_year"].notna()
            & (df["folder_year"] != df["date_year"]))
    for _, r in df[mism].iterrows():
        print(f"  [year mismatch] {r['source_file']}: folder={r['folder_year']} "
              f"dateline={r['date_year']}")

    df = df[["source_file", "folder_year", "date", "doc_type", "raw_text"]]
    df = df.sort_values("date", na_position="last").reset_index(drop=True)
    print(f"[clean] {len(df)} releases with an isolated body")
    return df

ecb_docs = clean_ecb_frame(ecb_raw)

# save the ECB equivalent of the FOMC csv into out/
OUT_DIR.mkdir(parents=True, exist_ok=True)
ecb_docs.to_csv(OUT_DIR / "ecb_statements_clean.csv", index=False)
ecb_docs[["source_file", "date", "raw_text"]].to_json(
    OUT_DIR / "ecb_statements_clean.jsonl",
    orient="records", lines=True, force_ascii=False,   # keep €/–/curly quotes readable
)
print(f"saved -> {OUT_DIR/'ecb_statements_clean.csv'}")
ecb_docs.head(10)

# %% [markdown]
# ## Sentence stage (self-contained): split -> Panel A1/B1 filter -> connective split
#
# This block defines EVERYTHING it needs, so it runs on top of the merge/clean
# cells above with nothing else in scope. `clean_text`, the base splitter, the
# Gorodnichenko panels, `TARGET_RE` / `TABLE1_RE` and `split_on_connectives` are
# copied verbatim from your FOMC notebook so the ECB corpus is filtered by the
# IDENTICAL dictionary and connective rules — that's what keeps the two banks
# comparable. If you later change the dictionary on the FOMC side, mirror it here.
#
# The only ECB-specific piece is `split_blocks`: an ECB page puts each paragraph
# and each subsection header on its own line, so a newline is a real sentence
# boundary (unlike your FOMC PDF text, where a line break is a mid-sentence wrap
# that `clean_text` collapses). It splits on newlines FIRST, then runs the base
# splitter within each block. Without it, modern releases glue a header onto the
# next sentence ("Refinancing operations As banks are repaying ..."); with it, a
# bare header becomes its own fragment and the Panel A1/B1 filter drops it.
#
# COVERAGE NOTE: the ECB called its headline rate the "minimum bid rate" pre-2008
# and the "interest rate on the main refinancing operations" from 2008 on. Panel
# A1 has interest/bank/fund rate but not "bid rate", so the main-rate sentence in
# 1999-2007 releases won't pass the filter (the marginal-lending and deposit
# "interest rate" sentences still do). Add "bid rate" to PANEL_A1 if you want
# them — left as-is here since the dictionary is your call. No FOMC boilerplate
# guard is included: extract_ecb_body already removed the page chrome, so there's
# nothing for it to catch.

# %%
import unicodedata

# ---- copied verbatim from the FOMC notebook (so this file is standalone) ----
def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKC", s)
    s = (s.replace("\u201c", '"').replace("\u201d", '"').replace("\u2018", "'")
           .replace("\u2019", "'").replace("\u2013", "-").replace("\u2014", "-")
           .replace("\u00a0", " "))
    s = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", s)   # "infla-\ntion" -> "inflation"
    s = re.sub(r"\s*\n\s*", " ", s)                 # (no-op per line; here for parity)
    s = re.sub(r"[ \t]+", " ", s).strip()
    return s

def split(text):
    """Base .!? sentence splitter (the get_splitter() fallback from your notebook)."""
    return re.split(r"(?<=[.!?])\s+", text) if isinstance(text, str) else []

PANEL_A1 = ["inflation expectation", "interest rate", "bank rate", "fund rate",
            "price", "economic activity", "inflation", "employment"]
PANEL_B1 = ["unemployment", "growth", "exchange rate", "productivity",
            "deficit", "demand", "job market", "monetary policy"]
PANEL_A2 = ["anchor", "cut", "subdue", "decline", "decrease", "reduce", "low",
            "drop", "fall", "fell", "decelerate", "slow", "pause", "pausing",
            "stable", "non-accelerating", "downward", "tighten"]
PANEL_B2 = ["ease", "easing", "rise", "rising", "increase", "expand", "improve",
            "strong", "upward", "raise", "high", "rapid"]
PANEL_C  = ["weren't", "were not", "wasn't", "was not", "did not", "didn't",
            "do not", "don't", "will not", "won't"]
USE_PREFIX_MATCH = False

def _dict_regex(terms, prefix=USE_PREFIX_MATCH):
    parts = [r"\s+".join(re.escape(w) for w in t.split())
             for t in sorted(terms, key=len, reverse=True)]
    tail = r"[A-Za-z]*" if prefix else ""
    return re.compile(
        r"(?<![A-Za-z])(?:" + "|".join(parts) + r")" + tail + r"(?![A-Za-z])",
        re.IGNORECASE)

TARGET_RE = _dict_regex(PANEL_A1 + PANEL_B1)                                 # filter
TABLE1_RE = _dict_regex(PANEL_A1 + PANEL_B1 + PANEL_A2 + PANEL_B2 + PANEL_C)  # split validity

def is_target_sentence(s):
    """Keep iff the sentence has a Panel A1/B1 term."""
    return bool(TARGET_RE.search(s)) if isinstance(s, str) else False

SPLIT_RE = re.compile(
    r"\s*(?:;|(?<![A-Za-z])(?:but|however|even\s+though|although|while)(?![A-Za-z]))\s*",
    re.IGNORECASE)

def split_on_connectives(sentence):
    """Split at connectives; keep the split only if EVERY segment is on-topic."""
    if not isinstance(sentence, str):
        return []
    segments = [seg.strip() for seg in SPLIT_RE.split(sentence) if seg and seg.strip()]
    if len(segments) <= 1:
        return [sentence.strip()]
    if all(TABLE1_RE.search(seg) for seg in segments):
        return segments
    return [sentence.strip()]

# ---- ECB-specific: newline-aware base split --------------------------------
def split_blocks(body):
    """clean_text + base split, applied PER LINE so ECB paragraph/header
    boundaries survive as sentence boundaries."""
    if not isinstance(body, str):
        return []
    out = []
    for block in re.split(r"\n+", body):     # ECB: newline = hard boundary
        block = clean_text(block)
        if block:
            out.extend(split(block))
    return out

# %%
# (a) split each isolated body into sentences (newline-aware)
ecb_docs["sentences"] = ecb_docs["raw_text"].map(split_blocks)

# (b) explode to one sentence per row
ecb_sents = (
    ecb_docs.explode("sentences", ignore_index=True)
            .rename(columns={"sentences": "sentence"})
            .dropna(subset=["sentence"])
)
ecb_sents["sentence"] = ecb_sents["sentence"].astype(str).str.strip()
ecb_sents = ecb_sents[ecb_sents["sentence"].astype(bool)].reset_index(drop=True)

# (c) Data & Title Filtration: keep only Panel A1/B1 sentences
before = len(ecb_sents)
ecb_sents = ecb_sents[ecb_sents["sentence"].map(is_target_sentence)].reset_index(drop=True)
print(f"filtration kept {len(ecb_sents)} / {before} sentences (Panel A1/B1 terms)")

# (d) connective split (kept only if EVERY segment is on-topic)
n_pre = len(ecb_sents)
ecb_sents["sentence"] = ecb_sents["sentence"].map(split_on_connectives)
ecb_sents = ecb_sents.explode("sentence", ignore_index=True)
ecb_sents["sentence"] = ecb_sents["sentence"].astype(str).str.strip()
ecb_sents = ecb_sents[ecb_sents["sentence"].astype(bool)].reset_index(drop=True)
print(f"splitting: {n_pre} target sentences -> {len(ecb_sents)} segments")

# (e) save the ECB sentence set
ecb_sents.to_csv(OUT_DIR / "ecb_sentences.csv", index=False)
print(f"saved {len(ecb_sents)} sentences -> {OUT_DIR / 'ecb_sentences.csv'}")
ecb_sents[["date", "doc_type", "sentence"]].head(20)

# %% [markdown]
# ### Holdout for manual labeling
#
# `ecb_sents` has `doc_type` + `sentence`, the same columns your FOMC holdout cell
# expects, so that cell works unchanged — point it at `ecb_sents`, keep
# `DOC_TYPE = "statement"`, set its `OUT_DIR` to this ECB `out/`, and use
# ECB-specific filenames so the labels never mix with the FED set. If you dedup,
# do it on `ecb_sents` BEFORE sampling the holdout (repeated stock phrasing recurs
# across meetings).

[merge] 314 raw .txt files from /Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/ECB BRANCH/ECB Monetary policy decisions/txt
  [year mismatch] 2020/2020-03-12-ecb-mp.txt: folder=2020 dateline=2019
[clean] 314 releases with an isolated body
saved -> /Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/ECB BRANCH/ECB Monetary policy decisions/out/ecb_statements_clean.csv
filtration kept 1033 / 1935 sentences (Panel A1/B1 terms)
splitting: 1033 target sentences -> 1058 segments
saved 1058 sentences -> /Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/ECB BRANCH/ECB Monetary policy decisions/out/ecb_sentences.csv


,date,doc_type,sentence
0,1999-03-04,statement,Taking into account all the information availa...
1,1999-03-04,statement,they will be conducted as fixed rate tenders a...
2,1999-03-04,statement,"In addition, the interest rate on the marginal..."
3,1999-03-18,statement,At today's meeting the Governing Council of th...
4,1999-03-18,statement,The Governing Council confirmed its earlier as...
5,1999-03-18,statement,they will be fixed rate tenders conducted at a...
6,1999-03-18,statement,"In addition, the interest rate on the marginal..."
7,1999-04-08,statement,At today's meeting the Governing Council of th...
8,1999-04-08,statement,The interest rate on the main refinancing oper...
9,1999-04-08,statement,The interest rate on the marginal lending faci...


In [ ]:
ecb_sents

In [6]:
# %% [markdown]
# ## ECB monetary-policy releases: merge all year folders -> clean
#
# The scraped releases live one-per-`.txt` inside per-year subfolders
# (`txt/1999/`, `txt/2000/`, ... `txt/2026/`), so this runs in two explicit steps:
#
#   STEP 1  MERGE  -> walk every year folder, read all raw .txt into ONE frame
#   STEP 2  CLEAN  -> isolate the monetary-policy body + parse the date
#
# After step 2 the frame has the SAME shape as the FOMC `docs` (`raw_text` +
# `doc_type`), so it drops straight into the downstream you already built
# (clean_text -> sentence split -> Gorodnichenko Panel A1/B1 filter -> holdout).
#
# Every scraped page has the same constant layout, 1999 through 2026:
#
#     [skip-links / search chrome / top nav menu]      <- junk (above the title)
#     PRESS RELEASE
#     Monetary policy decisions                        <- title (anchor)
#     <dd Month yyyy>                                  <- dateline (body starts AFTER)
#     ... the actual monetary-policy release ...       <- KEEP THIS
#     The President of the ECB will comment ...         <- procedural sign-off (body ENDS here)
#     Related topics / SEE ALSO / CONTACT / cookies    <- junk (below the body)

# %%
import re
from datetime import datetime
from pathlib import Path
import pandas as pd

# %% [markdown]
# ### !! Set this to wherever the "ECB Monetary policy decisions" folder lives !!
# `txt/` (the year subfolders) and `out/` are derived from it.

# %%
ECB_ROOT = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/"
                "ECB BRANCH/ECB Monetary policy decisions")

TXT_ROOT = ECB_ROOT / "txt"     # contains the 1999..2026 subfolders
OUT_DIR  = ECB_ROOT / "out"     # cleaned csv / jsonl written here

# Kept "statement" so this frame is compatible with your FOMC holdout sampler
# (it filters on doc_type == "statement"). Switch to "ecb_statement" only if you
# intend to CONCAT ECB with FOMC into one frame and want the banks separable.
ECB_DOC_TYPE = "statement"

# %% [markdown]
# ### Extraction patterns + functions

# %%
_MONTHS = (r"(?:January|February|March|April|May|June|July|August|"
           r"September|October|November|December)")

# The dateline directly under "Monetary policy decisions" — the handoff from page
# chrome to release body. "in the first half of 2000" won't match (needs
# day+month+year), so the first hit is always the real dateline.
ECB_DATE_RE = re.compile(rf"\b(\d{{1,2}}\s+{_MONTHS}\s+\d{{4}})\b", re.IGNORECASE)

# Body TERMINATORS — earliest one wins. The "President will comment ..." sign-off
# is a terminator, not noise: it's always the last line of the release, and on
# some pages (e.g. 6 June 2002) it's broken mid-sentence across three lines
# ("... at a" / "press conference" / "starting at 2.30 p.m. today"), so cutting
# here also removes those dangling fragments. Everything below (Related topics
# tag-list, SEE ALSO, CONTACT block, cookie/privacy notices) is page furniture.
# Anchored to line starts so a stray "contact" inside prose can't clip the body.
ECB_END_MARKERS = re.compile(
    r"^[ \t]*(?:"
    r"the president of the ecb will comment"     # procedural sign-off
    r"|related topics"                           # tag list (newer releases)
    r"|see also"                                 # 'find out more' block
    r"|contact[ \t]*$"                           # ECB address block header
    r"|media contacts"
    r"|disclaimer[ \t]*$"
    r"|are you happy with this page"
    r"|our website uses cookies"
    r")",
    re.IGNORECASE | re.MULTILINE,
)

# Scraped skip-links / search widget / top navigation menu items. Matched as
# WHOLE lines (exact, case-insensitive) so they can never clip real prose.
ECB_NAV_LINES = {
    "skip to:", "skip to", "navigation", "content", "footer",
    "anytime", "past month", "past year", "search options", "image preview",
    "home", "media", "explainers", "research & publications", "statistics",
    "monetary policy", "the \u20acuro", "payments & markets", "careers",
    "suggestions", "sort by", "relevance", "date", "press release",
    "monetary policy decisions",
}

# In-body leftover to drop line-wise (NOT a terminator): the '***' rule the ECB
# puts between the release and its closing paragraph is genuine mid-body markup,
# and the paragraph after it ("The Governing Council stands ready ...") is real
# content, so we delete only the separator line and keep going.
ECB_BODY_NOISE = re.compile(r"^\s*\*{2,}\s*$")


def extract_ecb_body(raw_text: str) -> str:
    """Return only the monetary-policy release text, stripped of page chrome."""
    if not isinstance(raw_text, str) or not raw_text.strip():
        return ""
    text = raw_text

    # 1) cut the footer at the earliest terminator
    end = ECB_END_MARKERS.search(text)
    if end:
        text = text[:end.start()]

    # 2) start after the dateline that follows the release title. Anchoring on the
    #    FIRST "monetary policy decisions" means a body that repeats the phrase
    #    (e.g. "took the following monetary policy decisions:") is left untouched.
    title = re.search(r"monetary policy decisions", text, re.IGNORECASE)
    if title:
        text = text[title.end():]
        d = ECB_DATE_RE.search(text)
        if d:
            text = text[d.end():]

    # 3) tidy surviving lines: drop nav tokens and '***' separators, keep prose
    kept = []
    for ln in text.splitlines():
        s = ln.strip()
        if not s or s.lower() in ECB_NAV_LINES or ECB_BODY_NOISE.match(s):
            continue
        kept.append(s)
    return "\n".join(kept).strip()


def parse_ecb_date(raw_text: str):
    """Dateline under the title, as YYYY-MM-DD (matches the FOMC date format)."""
    if not isinstance(raw_text, str):
        return None
    title = re.search(r"monetary policy decisions", raw_text, re.IGNORECASE)
    d = ECB_DATE_RE.search(raw_text, title.end() if title else 0)
    if not d:
        return None
    try:
        return datetime.strptime(d.group(1), "%d %B %Y").strftime("%Y-%m-%d")
    except ValueError:
        return None

# %% [markdown]
# ### STEP 1 - MERGE every year folder into one raw frame

# %%
_YEAR_RE = re.compile(r"(?:19|20)\d{2}")

def gather_ecb_txt(txt_root, pattern="*.txt") -> pd.DataFrame:
    """Recurse into every year subfolder and read each raw .txt into one frame.
    No cleaning yet, so you can eyeball the merged pile first. `source_file` is
    the path relative to txt_root (e.g. '2007/xxx.txt') so identically-named
    files in different year folders never collide."""
    txt_root = Path(txt_root)
    if not txt_root.exists():
        raise FileNotFoundError(f"{txt_root} not found — check ECB_ROOT.")
    rows = []
    for fp in sorted(txt_root.rglob(pattern)):        # rglob = descend into 1999..2026
        rel = fp.relative_to(txt_root).as_posix()
        ym = _YEAR_RE.search(rel)                     # year from the folder path
        rows.append({
            "source_file": rel,
            "folder_year": int(ym.group()) if ym else None,
            "raw_text_full": fp.read_text(encoding="utf-8", errors="replace"),
        })
    df = pd.DataFrame(rows, columns=["source_file", "folder_year", "raw_text_full"])
    print(f"[merge] {len(df)} raw .txt files from {txt_root}")
    return df

ecb_raw = gather_ecb_txt(TXT_ROOT)
ecb_raw.head()

# %% [markdown]
# ### STEP 2 - CLEAN the merged frame (isolate body + parse date)

# %%
def clean_ecb_frame(raw_df, doc_type=ECB_DOC_TYPE) -> pd.DataFrame:
    """Apply body isolation + date parse to the merged frame. Output is shaped
    like the FOMC `docs` (raw_text + doc_type) plus date / folder_year /
    source_file. Cross-checks dateline year vs folder year and warns on any
    mismatch (a misfiled or misparsed document)."""
    df = raw_df.copy()
    df["raw_text"]  = df["raw_text_full"].map(extract_ecb_body)
    df["date"]      = df["raw_text_full"].map(parse_ecb_date)
    df["doc_type"]  = doc_type
    df["date_year"] = df["date"].str.slice(0, 4).astype("Int64")

    # drop pages we couldn't isolate a body from (flag them, never guess)
    empty = df["raw_text"].str.len() == 0
    for f in df.loc[empty, "source_file"]:
        print(f"  [skip] no body extracted: {f}")
    df = df[~empty].copy()

    # folder-year vs dateline-year cross-check
    mism = (df["folder_year"].notna() & df["date_year"].notna()
            & (df["folder_year"] != df["date_year"]))
    for _, r in df[mism].iterrows():
        print(f"  [year mismatch] {r['source_file']}: folder={r['folder_year']} "
              f"dateline={r['date_year']}")

    df = df[["source_file", "folder_year", "date", "doc_type", "raw_text"]]
    df = df.sort_values("date", na_position="last").reset_index(drop=True)
    print(f"[clean] {len(df)} releases with an isolated body")
    return df

ecb_docs = clean_ecb_frame(ecb_raw)

# save the ECB equivalent of the FOMC csv into out/
OUT_DIR.mkdir(parents=True, exist_ok=True)
ecb_docs.to_csv(OUT_DIR / "ecb_statements_clean.csv", index=False)
ecb_docs[["source_file", "date", "raw_text"]].to_json(
    OUT_DIR / "ecb_statements_clean.jsonl",
    orient="records", lines=True, force_ascii=False,   # keep €/–/curly quotes readable
)
print(f"saved -> {OUT_DIR/'ecb_statements_clean.csv'}")
ecb_docs.head(10)

# %% [markdown]
# ## Sentence stage (self-contained): split -> Panel A1/B1 filter -> connective split
#
# This block defines EVERYTHING it needs, so it runs on top of the merge/clean
# cells above with nothing else in scope. `clean_text`, the base splitter, the
# Gorodnichenko panels, `TARGET_RE` / `TABLE1_RE` and `split_on_connectives` are
# copied verbatim from your FOMC notebook so the ECB corpus is filtered by the
# IDENTICAL dictionary and connective rules — that's what keeps the two banks
# comparable. If you later change the dictionary on the FOMC side, mirror it here.
#
# The only ECB-specific piece is `split_blocks`: an ECB page puts each paragraph
# and each subsection header on its own line, so a newline is a real sentence
# boundary (unlike your FOMC PDF text, where a line break is a mid-sentence wrap
# that `clean_text` collapses). It splits on newlines FIRST, then runs the base
# splitter within each block. Without it, modern releases glue a header onto the
# next sentence ("Refinancing operations As banks are repaying ..."); with it, a
# bare header becomes its own fragment and the Panel A1/B1 filter drops it.
#
# COVERAGE NOTE: the ECB called its headline rate the "minimum bid rate" pre-2008
# and the "interest rate on the main refinancing operations" from 2008 on. Panel
# A1 has interest/bank/fund rate but not "bid rate", so the main-rate sentence in
# 1999-2007 releases won't pass the filter (the marginal-lending and deposit
# "interest rate" sentences still do). Add "bid rate" to PANEL_A1 if you want
# them — left as-is here since the dictionary is your call. No FOMC boilerplate
# guard is included: extract_ecb_body already removed the page chrome, so there's
# nothing for it to catch.

# %%
import unicodedata

# ---- copied verbatim from the FOMC notebook (so this file is standalone) ----
def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKC", s)
    s = (s.replace("\u201c", '"').replace("\u201d", '"').replace("\u2018", "'")
           .replace("\u2019", "'").replace("\u2013", "-").replace("\u2014", "-")
           .replace("\u00a0", " "))
    s = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", s)   # "infla-\ntion" -> "inflation"
    s = re.sub(r"\s*\n\s*", " ", s)                 # (no-op per line; here for parity)
    s = re.sub(r"[ \t]+", " ", s).strip()
    return s

def split(text):
    """Base .!? sentence splitter (the get_splitter() fallback from your notebook)."""
    return re.split(r"(?<=[.!?])\s+", text) if isinstance(text, str) else []

PANEL_A1 = ["inflation expectation", "interest rate", "bank rate", "fund rate",
            "price", "economic activity", "inflation", "employment"]
PANEL_B1 = ["unemployment", "growth", "exchange rate", "productivity",
            "deficit", "demand", "job market", "monetary policy"]
PANEL_A2 = ["anchor", "cut", "subdue", "decline", "decrease", "reduce", "low",
            "drop", "fall", "fell", "decelerate", "slow", "pause", "pausing",
            "stable", "non-accelerating", "downward", "tighten"]
PANEL_B2 = ["ease", "easing", "rise", "rising", "increase", "expand", "improve",
            "strong", "upward", "raise", "high", "rapid"]
PANEL_C  = ["weren't", "were not", "wasn't", "was not", "did not", "didn't",
            "do not", "don't", "will not", "won't"]
USE_PREFIX_MATCH = False

def _dict_regex(terms, prefix=USE_PREFIX_MATCH):
    parts = [r"\s+".join(re.escape(w) for w in t.split())
             for t in sorted(terms, key=len, reverse=True)]
    tail = r"[A-Za-z]*" if prefix else ""
    return re.compile(
        r"(?<![A-Za-z])(?:" + "|".join(parts) + r")" + tail + r"(?![A-Za-z])",
        re.IGNORECASE)

TARGET_RE = _dict_regex(PANEL_A1 + PANEL_B1)                                 # filter
TABLE1_RE = _dict_regex(PANEL_A1 + PANEL_B1 + PANEL_A2 + PANEL_B2 + PANEL_C)  # split validity

def is_target_sentence(s):
    """Keep iff the sentence has a Panel A1/B1 term."""
    return bool(TARGET_RE.search(s)) if isinstance(s, str) else False

SPLIT_RE = re.compile(
    r"\s*(?:;|(?<![A-Za-z])(?:but|however|even\s+though|although|while)(?![A-Za-z]))\s*",
    re.IGNORECASE)

def split_on_connectives(sentence):
    """Split at connectives; keep the split only if EVERY segment is on-topic."""
    if not isinstance(sentence, str):
        return []
    segments = [seg.strip() for seg in SPLIT_RE.split(sentence) if seg and seg.strip()]
    if len(segments) <= 1:
        return [sentence.strip()]
    if all(TABLE1_RE.search(seg) for seg in segments):
        return segments
    return [sentence.strip()]

# ---- ECB-specific: newline-aware base split --------------------------------
def split_blocks(body):
    """clean_text + base split, applied PER LINE so ECB paragraph/header
    boundaries survive as sentence boundaries."""
    if not isinstance(body, str):
        return []
    out = []
    for block in re.split(r"\n+", body):     # ECB: newline = hard boundary
        block = clean_text(block)
        if block:
            out.extend(split(block))
    return out

# %%
# (a) split each isolated body into sentences (newline-aware)
ecb_docs["sentences"] = ecb_docs["raw_text"].map(split_blocks)

# (b) explode to one sentence per row
ecb_sents = (
    ecb_docs.explode("sentences", ignore_index=True)
            .rename(columns={"sentences": "sentence"})
            .dropna(subset=["sentence"])
)
ecb_sents["sentence"] = ecb_sents["sentence"].astype(str).str.strip()
ecb_sents = ecb_sents[ecb_sents["sentence"].astype(bool)].reset_index(drop=True)

# (c) Data & Title Filtration: keep only Panel A1/B1 sentences
before = len(ecb_sents)
ecb_sents = ecb_sents[ecb_sents["sentence"].map(is_target_sentence)].reset_index(drop=True)
print(f"filtration kept {len(ecb_sents)} / {before} sentences (Panel A1/B1 terms)")

# (d) connective split (kept only if EVERY segment is on-topic)
n_pre = len(ecb_sents)
ecb_sents["sentence"] = ecb_sents["sentence"].map(split_on_connectives)
ecb_sents = ecb_sents.explode("sentence", ignore_index=True)
ecb_sents["sentence"] = ecb_sents["sentence"].astype(str).str.strip()
ecb_sents = ecb_sents[ecb_sents["sentence"].astype(bool)].reset_index(drop=True)
print(f"splitting: {n_pre} target sentences -> {len(ecb_sents)} segments")

# (e) save the ECB sentence set
ecb_sents.to_csv(OUT_DIR / "ecb_sentences.csv", index=False)
print(f"saved {len(ecb_sents)} sentences -> {OUT_DIR / 'ecb_sentences.csv'}")
ecb_sents[["date", "doc_type", "sentence"]].head(20)

# %% [markdown]
# ## Train / test split
#
# Randomly pick 800 sentences for TRAIN; everything else goes to TEST. The split
# is reproducible (fixed seed), non-mutating, and guarded so a re-run can't
# silently produce a DIFFERENT split and clobber labels you've started (same
# safeguard as your FOMC holdout). Each row keeps a stable `id` (its ecb_sents
# row) so you can trace it back, plus an empty `label` column to fill by hand.
#
# If ecb_sents has <= 800 rows the split takes the min: train gets everything and
# test is empty (won't happen on the full archive; guaranteed on the tiny demo).

# %%
N_TRAIN   = 800
SEED      = 42
OVERWRITE = False   # flip to True ONLY when you deliberately want a fresh split

def make_train_test(df, n_train, seed):
    """Return (train, test) WITHOUT mutating df. Deterministic for a given seed."""
    n_take = min(int(n_train), len(df))
    train = df.sample(n=n_take, random_state=seed)
    test  = df.drop(index=train.index)
    return train, test

def _prep(frame):
    """Add a stable id (original ecb_sents row) + empty label column for annotation."""
    out = frame.copy()
    out.insert(0, "id", out.index)                       # trace back to ecb_sents
    out["label"] = ""                                     # <- fill in by hand
    cols = ["id", "source_file", "date", "doc_type", "sentence", "label"]
    return out[[c for c in cols if c in out.columns]].reset_index(drop=True)

train_df, test_df = make_train_test(ecb_sents, N_TRAIN, SEED)
train_out, test_out = _prep(train_df), _prep(test_df)

# paths + overwrite guard (checked BEFORE writing anything)
train_csv = OUT_DIR / "ecb_train.csv"
test_csv  = OUT_DIR / "ecb_test.csv"
if train_csv.exists() and not OVERWRITE:
    raise FileExistsError(
        f"{train_csv} already exists. Regenerating would create a DIFFERENT split "
        f"and discard any labels you've started. Set OVERWRITE=True to proceed.")

OUT_DIR.mkdir(parents=True, exist_ok=True)
train_out.to_csv(train_csv, index=False)
test_out.to_csv(test_csv, index=False)
train_out[["id", "sentence", "label"]].to_json(
    OUT_DIR / "ecb_train.jsonl", orient="records", lines=True, force_ascii=False)
test_out[["id", "sentence", "label"]].to_json(
    OUT_DIR / "ecb_test.jsonl", orient="records", lines=True, force_ascii=False)

print(f"train: {len(train_out)} sentences -> {train_csv.name} / ecb_train.jsonl")
print(f"test:  {len(test_out)} sentences -> {test_csv.name} / ecb_test.jsonl")

[merge] 314 raw .txt files from /Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/ECB BRANCH/ECB Monetary policy decisions/txt
  [year mismatch] 2020/2020-03-12-ecb-mp.txt: folder=2020 dateline=2019
[clean] 314 releases with an isolated body
saved -> /Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/ECB BRANCH/ECB Monetary policy decisions/out/ecb_statements_clean.csv
filtration kept 1033 / 1935 sentences (Panel A1/B1 terms)
splitting: 1033 target sentences -> 1058 segments
saved 1058 sentences -> /Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/ECB BRANCH/ECB Monetary policy decisions/out/ecb_sentences.csv
train: 800 sentences -> ecb_train.csv / ecb_train.jsonl
test:  258 sentences -> ecb_test.csv / ecb_test.jsonl
